In [1]:
import pandas as pd 
import numpy as np

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
data=pd.read_csv("mentalhealthdataset.csv")

In [5]:
data['self_employed'].value_counts(dropna=False)

self_employed
No     257994
Yes     29168
NaN      5202
Name: count, dtype: int64

In [7]:
data['self_employed']=data['self_employed'].fillna('No')

In [11]:
data = data.drop_duplicates()

In [12]:
def define_risk(row):
    
    if (row['treatment'] == 'Yes' or
        (row['Mental_Health_History'] == 'Yes' and row['Mood_Swings'] == 'High')):
        return 'High'
        
    if (row['treatment'] == 'No' and
        row['Mental_Health_History'] == 'No' and
        row['Mood_Swings'] == 'Low'):
        return 'Low'
    
    return 'Medium'

data['Risk_Level'] = data.apply(define_risk, axis=1)
print(data['Risk_Level'].value_counts())

Risk_Level
High      160276
Medium    112509
Low        17266
Name: count, dtype: int64


In [8]:
# 1. Define the file path for the new Excel file
#excel_file_path = 'mental_health_data_with_risk_levels.xlsx' 

# 2. Export the DataFrame to Excel
# index=False prevents pandas from writing the DataFrame's index (row numbers) as a column.
#try:
    #data.to_excel(excel_file_path, index=False)
    #print(f"\n✅ Success! The updated DataFrame has been saved to: {excel_file_path}")
    #print("\nFile contents include the new 'Risk_Level' column.")

#except Exception as e:
    #print(f"\n❌ An error occurred while saving to Excel: {e}")
    #print("Please ensure you have the 'openpyxl' library installed (pip install openpyxl).")

In [15]:
from sklearn.preprocessing import LabelEncoder
binary_cols=['Gender','self_employed','family_history','treatment','Coping_Struggles']
for col in binary_cols:
    data[col]=LabelEncoder().fit_transform(data[col])

In [16]:
data['Days_Indoors']= data['Days_Indoors'].map({'Go out Every day': 0, '1-14 days': 1, '15-30 days': 2, '31-60 days': 3, 'More than 2 months': 4})

In [19]:
data['Growing_Stress'] = data['Growing_Stress'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Changes_Habits'] = data['Changes_Habits'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Mental_Health_History'] = data['Mental_Health_History'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Mood_Swings'] = data['Mood_Swings'].map({'Low': 0, 'Medium': 1, 'High': 2})
data['Work_Interest'] = data['Work_Interest'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Social_Weakness'] = data['Social_Weakness'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['mental_health_interview'] = data['mental_health_interview'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['care_options'] = data['care_options'].map({'No': 0, 'Maybe': 1, 'Yes': 2, 'Not sure': 1})

In [21]:
data = pd.get_dummies(data, columns=['Country', 'Occupation'], drop_first=True)

In [23]:
data=data.drop(columns=['Timestamp'])
data.head()

,Gender,self_employed,family_history,treatment,Days_Indoors,Growing_Stress,Changes_Habits,Mental_Health_History,Mood_Swings,Coping_Struggles,...,Country_South Africa,Country_Sweden,Country_Switzerland,Country_Thailand,Country_United Kingdom,Country_United States,Occupation_Corporate,Occupation_Housewife,Occupation_Others,Occupation_Student
0,0,0,0,1,1,2,0,2,1,0,...,False,False,False,False,False,True,True,False,False,False
1,0,0,1,1,1,2,0,2,1,0,...,False,False,False,False,False,True,True,False,False,False
2,0,0,1,1,1,2,0,2,1,0,...,False,False,False,False,False,True,True,False,False,False
3,0,0,1,1,1,2,0,2,1,0,...,False,False,False,False,False,True,True,False,False,False
4,0,0,1,1,1,2,0,2,1,0,...,False,False,False,False,False,True,True,False,False,False


In [25]:
bool_cols = data.select_dtypes('bool').columns
data[bool_cols] = data[bool_cols].astype(int)

In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE


X = data.drop(columns=['Risk_Level', 'treatment'])
y = data['Risk_Level']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)




rf_model = RandomForestClassifier(n_estimators=100,max_depth=15,random_state=42,class_weight='balanced')
rf_model.fit(X_test, y_test)

y_pred = rf_model.predict(X_test)

print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

        High       0.85      0.72      0.78     48083
         Low       0.57      1.00      0.72      5180
      Medium       0.74      0.82      0.78     33753

    accuracy                           0.77     87016
   macro avg       0.72      0.84      0.76     87016
weighted avg       0.79      0.77      0.77     87016

Confusion Matrix:
 [[34515  3966  9602]
 [    8  5172     0]
 [ 6129     0 27624]]


In [29]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

# ✅ Correcting: model should be trained on training data
X = data.drop(columns=['Risk_Level', 'treatment'])
y = data['Risk_Level']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Create and train MLP model
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),   # two hidden layers
    activation='relu',              # ReLU activation
    solver='adam',                  # Adam optimizer
    max_iter=500,                   # number of training iterations
    random_state=42,
    learning_rate='adaptive'        # adapt learning rate automatically
)

mlp_model.fit(X_train, y_train)

# Predictions
y_pred = mlp_model.predict(X_test)

# Evaluation
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

        High       0.79      0.85      0.82     48083
         Low       0.78      0.75      0.76      5180
      Medium       0.80      0.72      0.76     33753

    accuracy                           0.79     87016
   macro avg       0.79      0.77      0.78     87016
weighted avg       0.79      0.79      0.79     87016

Confusion Matrix:
 [[40850  1113  6120]
 [ 1313  3867     0]
 [ 9355     0 24398]]


In [31]:
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


X = data.drop(columns=['Risk_Level', 'treatment'])
y = data['Risk_Level']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)







gb_model = GradientBoostingClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42)


gb_model.fit(X_train, y_train)


y_pred = gb_model.predict(X_test)

print("\n--- Gradient Boosting Classification Report ---\n", classification_report(y_test, y_pred))
print("Gradient Boosting Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



--- Gradient Boosting Classification Report ---
               precision    recall  f1-score   support

        High       0.79      0.86      0.82     48083
         Low       0.79      0.72      0.75      5180
      Medium       0.80      0.71      0.76     33753

    accuracy                           0.79     87016
   macro avg       0.79      0.76      0.78     87016
weighted avg       0.79      0.79      0.79     87016

Gradient Boosting Confusion Matrix:
 [[41247   986  5850]
 [ 1456  3724     0]
 [ 9708     0 24045]]


In [62]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Note: The 'imblearn.over_sampling' SMOTE import is removed

import lightgbm as lgb

# Assuming 'data' DataFrame is available and preprocessed with the 'Risk_Level' column

# --- 1. Data Preparation (Same as before) ---
X = data.drop(columns=['Risk_Level', 'treatment'])
y = data['Risk_Level']

# --- 2. Train-Test Split (Same as before) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("--- Data Split Summary ---")
print("Original training distribution:\n", y_train.value_counts())
# The SMOTE resampling section is completely removed here

# --- 3. Target Variable Mapping (Crucial for LightGBM) ---
# Map the string labels to integers for LightGBM's multiclass objective
label_map = {'Low': 0, 'Medium': 1, 'High': 2}
y_train_lgbm = y_train.map(label_map)
y_test_lgbm = y_test.map(label_map) # Map the test set for comparison, though y_test is used in classification_report

# --- 4. LightGBM Model ---
lgbm_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=400,
    max_depth=20,
    learning_rate=0.05,
    metric='multi_logloss',
    random_state=42,
    n_jobs=-1
)

print("\nFitting LightGBM Model on Original Data (No SMOTE)...")
# Fit the model using the original training data (X_train and y_train_lgbm)
lgbm_model.fit(X_train, y_train_lgbm)

# --- 5. Prediction and Evaluation ---
y_pred_lgbm_int = lgbm_model.predict(X_test)

# Map predictions back to original labels for clear evaluation
reverse_label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
y_pred_lgbm = pd.Series(y_pred_lgbm_int).map(reverse_label_map)

print("\n--- LightGBM Classification Report (No SMOTE) ---\n", classification_report(y_test, y_pred_lgbm))
print("LightGBM Confusion Matrix (No SMOTE):\n", confusion_matrix(y_test, y_pred_lgbm))

--- Data Split Summary ---
Original training distribution:
 Risk_Level
High      112193
Medium     78756
Low        12086
Name: count, dtype: int64

Fitting LightGBM Model on Original Data (No SMOTE)...

--- LightGBM Classification Report (No SMOTE) ---
               precision    recall  f1-score   support

        High       0.79      0.86      0.82     48083
         Low       0.79      0.72      0.75      5180
      Medium       0.81      0.71      0.76     33753

    accuracy                           0.80     87016
   macro avg       0.80      0.77      0.78     87016
weighted avg       0.80      0.80      0.79     87016

LightGBM Confusion Matrix (No SMOTE):
 [[41478   990  5615]
 [ 1441  3739     0]
 [ 9726     0 24027]]


In [64]:
# --- 6. Combine test features, actual, and predicted labels ---
results_lgbm = X_test.copy()
results_lgbm['Actual_Risk_Level'] = y_test
results_lgbm['Predicted_Risk_Level'] = y_pred_lgbm

# --- 7. Filter correctly predicted Low class examples ---
correct_low = results_lgbm[
    (results_lgbm['Actual_Risk_Level'] == 'Low') &
    (results_lgbm['Predicted_Risk_Level'] == 'Low')
]

# Display first few correctly predicted Low examples
if not correct_low.empty:
    print("✅ Correctly predicted Low class examples (LightGBM):")
    print(correct_low.head(10))
else:
    print("⚠️ No correctly predicted Low class instances found.")

# --- 8. Optional: see all Low class predictions (correct and misclassified) ---
all_low = results_lgbm[results_lgbm['Actual_Risk_Level'] == 'Low']
print("\nAll Low class predictions (first 10 rows):")
print(all_low.head(10))


✅ Correctly predicted Low class examples (LightGBM):
       Gender  self_employed  family_history  Days_Indoors  Growing_Stress  \
80970       1              0               0             2               2   
85492       1              0               0             3               2   
26094       0              0               0             1               1   
85129       1              0               0             3               2   
51275       0              0               1             0               2   
11271       0              0               0             0               1   
8482        0              0               0             1               2   
61355       1              0               0             2               2   
462         0              0               1             0               2   
7638        0              0               0             2               1   

       Changes_Habits  Mental_Health_History  Mood_Swings  Coping_Struggles  \
80970    

In [19]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# --- 1. Ensure boolean columns are numeric ---
bool_cols = data.select_dtypes('bool').columns
data[bool_cols] = data[bool_cols].astype(int)

# --- 2. Data Preparation (Keep Mental_Health_History and Mood_Swings) ---
X = data.drop(columns=['Risk_Level', 'treatment'])  # only drop the target and treatment
y = data['Risk_Level']

# --- 3. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("--- Data Split Summary ---")
print("Training distribution:\n", y_train.value_counts())
print("Testing distribution:\n", y_test.value_counts())

# --- 4. Label Encoding for CatBoost ---
label_map = {'Low': 0, 'Medium': 1, 'High': 2}
y_train_catboost = y_train.map(label_map)
y_test_catboost = y_test.map(label_map)

# --- 5. CatBoost Model (No SMOTE) ---
catboost_model = CatBoostClassifier(
    iterations=400,          # number of trees
    depth=10,                # tree depth
    learning_rate=0.05,      # step size
    random_state=42,         # reproducibility
    verbose=200,             # print every 200 iterations
    loss_function='MultiClass'  # specify multiclass classification
)

print("\nFitting CatBoost Model on Original Data (No SMOTE)...")
catboost_model.fit(X_train, y_train_catboost)

# --- 6. Predictions ---
y_pred_catboost = catboost_model.predict(X_test).flatten()

# Map back to original class labels
reverse_label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
y_pred_catboost_labels = pd.Series(y_pred_catboost).map(reverse_label_map)

# --- 7. Evaluation ---
print("\n--- CatBoost Classification Report (No SMOTE) ---\n",
      classification_report(y_test, y_pred_catboost_labels))
print("CatBoost Confusion Matrix (No SMOTE):\n",
      confusion_matrix(y_test, y_pred_catboost_labels))


--- Data Split Summary ---
Training distribution:
 Risk_Level
High      112193
Medium     78756
Low        12086
Name: count, dtype: int64
Testing distribution:
 Risk_Level
High      48083
Medium    33753
Low        5180
Name: count, dtype: int64

Fitting CatBoost Model on Original Data (No SMOTE)...
0:	learn: 1.0525351	total: 293ms	remaining: 1m 56s
200:	learn: 0.4113092	total: 26.6s	remaining: 26.3s
399:	learn: 0.3910156	total: 49.3s	remaining: 0us

--- CatBoost Classification Report (No SMOTE) ---
               precision    recall  f1-score   support

        High       0.79      0.86      0.82     48083
         Low       0.80      0.72      0.76      5180
      Medium       0.81      0.71      0.76     33753

    accuracy                           0.80     87016
   macro avg       0.80      0.77      0.78     87016
weighted avg       0.80      0.80      0.79     87016

CatBoost Confusion Matrix (No SMOTE):
 [[41496   927  5660]
 [ 1443  3737     0]
 [ 9702     0 24051]]


In [66]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb

# --- 1. Define risk function (optional, if you need it for creating Risk_Level) ---
def define_risk(row):
    if (row['treatment'] == 1 or 
        (row['Mental_Health_History'] == 2 and row['Mood_Swings'] == 2)):
        return 'High'
    if (row['treatment'] == 0 and 
        row['Mental_Health_History'] == 0 and 
        row['Mood_Swings'] == 0):
        return 'Low'
    return 'Medium'

# --- 2. Ensure boolean columns are numeric ---
bool_cols = data.select_dtypes('bool').columns
data[bool_cols] = data[bool_cols].astype(int)

# --- 3. Data preparation (keep all important columns) ---
X = data.drop(columns=['Risk_Level', 'treatment'])  # only drop the target and treatment
y = data['Risk_Level']

# --- 4. Train-test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("--- Data Split Summary ---")
print("Training distribution:\n", y_train.value_counts())
print("Testing distribution:\n", y_test.value_counts())

# --- 5. Encode target labels for XGBoost ---
label_map = {'Low': 0, 'Medium': 1, 'High': 2}
y_train_xgb = y_train.map(label_map)
y_test_xgb = y_test.map(label_map)

# --- 6. XGBoost model (No SMOTE) ---
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',  # multiclass classification
    num_class=3,
    n_estimators=200,
    max_depth=10,
    learning_rate=0.05,
    eval_metric='merror',       # multiclass error rate
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

print("\nFitting XGBoost Model on Original Data (No SMOTE)...")
xgb_model.fit(X_train, y_train_xgb)

# --- 7. Predictions ---
y_pred_xgb_int = xgb_model.predict(X_test)

# Map predictions back to original labels
reverse_label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
y_pred_xgb = pd.Series(y_pred_xgb_int).map(reverse_label_map)

# --- 8. Evaluation ---
print("\n--- XGBoost Classification Report (No SMOTE) ---\n",
      classification_report(y_test, y_pred_xgb))
print("XGBoost Confusion Matrix (No SMOTE):\n",
      confusion_matrix(y_test, y_pred_xgb))


--- Data Split Summary ---
Training distribution:
 Risk_Level
High      112193
Medium     78756
Low        12086
Name: count, dtype: int64
Testing distribution:
 Risk_Level
High      48083
Medium    33753
Low        5180
Name: count, dtype: int64

Fitting XGBoost Model on Original Data (No SMOTE)...


C:\Users\user\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:35:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- XGBoost Classification Report (No SMOTE) ---
               precision    recall  f1-score   support

        High       0.79      0.87      0.82     48083
         Low       0.80      0.72      0.76      5180
      Medium       0.81      0.71      0.76     33753

    accuracy                           0.80     87016
   macro avg       0.80      0.76      0.78     87016
weighted avg       0.80      0.80      0.79     87016

XGBoost Confusion Matrix (No SMOTE):
 [[41621   919  5543]
 [ 1438  3742     0]
 [ 9919     0 23834]]


In [68]:
# --- 9. Combine test features, actual, and predicted labels ---
results_xgb = X_test.copy()
results_xgb['Actual_Risk_Level'] = y_test
results_xgb['Predicted_Risk_Level'] = y_pred_xgb

# --- 10. Filter correctly predicted Low class examples ---
correct_low_xgb = results_xgb[
    (results_xgb['Actual_Risk_Level'] == 'Low') &
    (results_xgb['Predicted_Risk_Level'] == 'Low')
]

# Display first few correctly predicted Low examples
if not correct_low_xgb.empty:
    print("✅ Correctly predicted Low class examples (XGBoost):")
    print(correct_low_xgb.head(10))
else:
    print("⚠️ No correctly predicted Low class instances found.")

# --- 11. Optional: see all Low class predictions (correct and misclassified) ---
all_low_xgb = results_xgb[results_xgb['Actual_Risk_Level'] == 'Low']
print("\nAll Low class predictions (first 10 rows):")
print(all_low_xgb.head(10))


✅ Correctly predicted Low class examples (XGBoost):
       Gender  self_employed  family_history  Days_Indoors  Growing_Stress  \
80970       1              0               0             2               2   
85492       1              0               0             3               2   
85129       1              0               0             3               2   
51275       0              0               1             0               2   
11271       0              0               0             0               1   
8482        0              0               0             1               2   
61355       1              0               0             2               2   
462         0              0               1             0               2   
85186       1              0               0             3               2   
8787        0              0               0             4               1   

       Changes_Habits  Mental_Health_History  Mood_Swings  Coping_Struggles  \
80970     

In [70]:
# --- Filter for a specific test instance ---
instance_index = 80970

if instance_index in X_test.index:
    instance = X_test.loc[[instance_index]].copy()  # keep as DataFrame
    instance['Actual_Risk_Level'] = y_test.loc[instance_index]
    instance['Predicted_Risk_Level'] = y_pred_xgb.loc[instance_index]
    
    print(f"Details for instance {instance_index}:")
    print(instance.T)  # transpose for easier viewing
else:
    print(f"⚠️ Instance {instance_index} not found in the test set.")


Details for instance 80970:
                               80970
Gender                             1
self_employed                      0
family_history                     0
Days_Indoors                       2
Growing_Stress                     2
Changes_Habits                     0
Mental_Health_History              0
Mood_Swings                        0
Coping_Struggles                   0
Work_Interest                      2
Social_Weakness                    2
mental_health_interview            1
care_options                       1
Country_Belgium                    0
Country_Bosnia and Herzegovina     0
Country_Brazil                     0
Country_Canada                     0
Country_Colombia                   0
Country_Costa Rica                 0
Country_Croatia                    0
Country_Czech Republic             0
Country_Denmark                    0
Country_Finland                    0
Country_France                     0
Country_Georgia                    0
Country_Ge

In [72]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- 1. Prepare features and target (include mental health features) ---
X = data.drop(columns=['Risk_Level', 'treatment'])  # Keep Mental_Health_History and Mood_Swings
y = data['Risk_Level']

# Encode target labels
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
class_names = le_target.classes_
print(f"Target classes encoded: {class_names} -> {le_target.transform(class_names)}")

# --- 2. Train-test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
print("--- Data Split Summary ---")
print("Training distribution:\n", pd.Series(y_train).value_counts())
print("Testing distribution:\n", pd.Series(y_test).value_counts())

# --- 3. Base models (no SMOTE) ---
lgbm_model = LGBMClassifier(
    objective='multiclass',
    num_class=len(class_names),
    n_estimators=400,
    max_depth=20,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(class_names),
    n_estimators=200,
    max_depth=10,
    learning_rate=0.05,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

catboost_model = CatBoostClassifier(
    iterations=400,
    depth=10,
    learning_rate=0.05,
    random_state=42,
    verbose=0,
    loss_function='MultiClass'
)

# --- 4. Stacked Ensemble ---
base_estimators = [
    ('lgbm', lgbm_model),
    ('xgb', xgb_model),
    ('catboost', catboost_model)
]

final_estimator = LogisticRegression(max_iter=1000, multi_class='multinomial')

stacked_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=final_estimator,
    cv=5,
    stack_method='predict_proba',
    n_jobs=-1
)

# --- 5. Train Stacked Ensemble ---
print("\nTraining Stacked Ensemble (No SMOTE, all features included)...")
stacked_model.fit(X_train, y_train)

# --- 6. Predictions & Evaluation ---
y_pred_stack = stacked_model.predict(X_test)

print("\nStacked Ensemble Accuracy:", accuracy_score(y_test, y_pred_stack))
print("\nStacked Ensemble Classification Report:\n",
      classification_report(y_test, y_pred_stack, target_names=class_names))
print("Stacked Ensemble Confusion Matrix:\n",
      confusion_matrix(y_test, y_pred_stack))

# --- 7. Optional: See predictions for Low class only ---
results_df = X_test.copy()
results_df['Actual_Risk_Level'] = le_target.inverse_transform(y_test)
results_df['Predicted_Risk_Level'] = le_target.inverse_transform(y_pred_stack)

low_class_predictions = results_df[results_df['Actual_Risk_Level'] == 'Low']
print("\nPredictions for Low class instances:\n", low_class_predictions.head(10))


Target classes encoded: ['High' 'Low' 'Medium'] -> [0 1 2]
--- Data Split Summary ---
Training distribution:
 0    112193
2     78756
1     12086
Name: count, dtype: int64
Testing distribution:
 0    48083
2    33753
1     5180
Name: count, dtype: int64

Training Stacked Ensemble (No SMOTE, all features included)...


C:\Users\user\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Stacked Ensemble Accuracy: 0.7987151788176887

Stacked Ensemble Classification Report:
               precision    recall  f1-score   support

        High       0.79      0.86      0.83     48083
         Low       0.81      0.71      0.76      5180
      Medium       0.81      0.72      0.76     33753

    accuracy                           0.80     87016
   macro avg       0.81      0.76      0.78     87016
weighted avg       0.80      0.80      0.80     87016

Stacked Ensemble Confusion Matrix:
 [[41567   840  5676]
 [ 1501  3679     0]
 [ 9498     0 24255]]

Predictions for Low class instances:
         Gender  self_employed  family_history  Days_Indoors  Growing_Stress  \
190935       1              0               0             4               2   
212891       1              0               0             0               0   
95785        1              0               1             3               1   
274071       1              0               0             1               1